# 03 — Training & Evaluation v2

Notebook này benchmark baseline training trên feature table `v2_rgb248_exact`.

- Input: `features/feature_extraction_v2_rgb248_exact.csv`
- Candidate sets: `control_minimal`, `always_on`, `always_on_plus_cfa_raw`, `always_on_plus_cfa_gated`, `full_v2`
- Models: `logreg`, `lightgbm`
- Selection: theo `val_auc`, calibration bằng calibration split, threshold khóa trên `val` với target `FPR <= 5%`

In [1]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent.resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.training import FEATURE_SET_COLUMNS, run_training_baseline

FEATURE_TABLE = PROJECT_ROOT / 'features' / 'feature_extraction_v2_rgb248_exact.csv'
RUN_NAME = 'training_v2_baseline_20260403'
OUTPUT_DIR = PROJECT_ROOT / 'audit_output' / 'validation' / RUN_NAME
FORCE_RERUN = os.getenv('TRAINING_V2_FORCE_RERUN', '0') == '1'
SUMMARY_PATH = OUTPUT_DIR / 'summary.json'
print({'feature_table': str(FEATURE_TABLE), 'output_dir': str(OUTPUT_DIR), 'feature_sets': list(FEATURE_SET_COLUMNS)})

{'feature_table': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\features\\feature_extraction_v2_rgb248_exact.csv', 'output_dir': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\audit_output\\validation\\training_v2_baseline_20260403', 'feature_sets': ['control_minimal', 'always_on', 'always_on_plus_cfa_raw', 'always_on_plus_cfa_gated', 'full_v2']}


## 1. Run or load baseline benchmark

Nếu artifact đã tồn tại và `TRAINING_V2_FORCE_RERUN=False`, notebook chỉ load lại. Nếu không, notebook chạy toàn bộ benchmark.

In [2]:
if SUMMARY_PATH.exists() and not FORCE_RERUN:
    summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
else:
    summary = run_training_baseline(FEATURE_TABLE, output_dir=OUTPUT_DIR)
summary

{'feature_table_path': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\features\\feature_extraction_v2_rgb248_exact.csv',
 'rows': 85615,
 'feature_version': 'v2_rgb248_exact_multibranch',
 'preprocess_version': 'v4_rgb248_r4_exact',
 'candidate_count': 10,
 'selected_candidate': 'full_v2__lightgbm',
 'selected_feature_set': 'full_v2',
 'selected_model_name': 'lightgbm',
 'selected_val_auc': 0.9548272480207964,
 'selected_val_brier': 0.08316389570949588,
 'selected_val_ece': 0.014494236795867962,
 'selected_threshold': 0.7074744498826763,
 'cfa_threshold': -0.5329408775144153,
 'split_role_counts': {'train_core': 44235,
  'ood_eval': 27410,
  'val': 5821,
  'id_test': 5821,
  'calibration': 2328},
 'files': {'candidate_metrics_csv': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\audit_output\\validation\\training_v2_baseline_20260403\\candidate_val_metrics.csv',
  'selected_metrics_csv': 'C:\\Users\\USER\\Desktop\\ai_detector_img\\audit_output\\validation\\training_v2_baseline_20260403\\selec

## 2. Candidate ranking on validation split

In [3]:
candidate_metrics = pd.read_csv(OUTPUT_DIR / 'candidate_val_metrics.csv')
candidate_metrics.sort_values(['val_auc', 'val_brier'], ascending=[False, True]).head(10)

,candidate_name,feature_set,model_name,model_family,feature_count,cfa_threshold,val_auc,val_brier,val_ece,val_threshold,val_tpr,val_fpr,val_precision,val_accuracy
0,full_v2__lightgbm,full_v2,lightgbm,tree,36,-0.532941,0.954827,0.083164,0.014494,0.707474,0.795333,0.049982,0.944203,0.870297
1,always_on_plus_cfa_raw__lightgbm,always_on_plus_cfa_raw,lightgbm,tree,19,-0.532941,0.933964,0.102545,0.012802,0.730323,0.726000,0.049982,0.939198,0.834565
2,full_v2__logreg,full_v2,logreg,linear,36,-0.532941,0.914035,0.116339,0.018238,0.715095,0.675667,0.049982,0.934963,0.808624
3,always_on_plus_cfa_raw__logreg,always_on_plus_cfa_raw,logreg,linear,19,-0.532941,0.904559,0.123456,0.015545,0.716501,0.652667,0.049982,0.932825,0.796770
4,always_on_plus_cfa_gated__lightgbm,always_on_plus_cfa_gated,lightgbm,tree,19,-0.532941,0.868869,0.147525,0.009757,0.776969,0.477000,0.049982,0.910305,0.706236
5,always_on__lightgbm,always_on,lightgbm,tree,14,-0.532941,0.821360,0.172383,0.018942,0.775546,0.384000,0.049982,0.890951,0.658306
6,always_on_plus_cfa_gated__logreg,always_on_plus_cfa_gated,logreg,linear,19,-0.532941,0.777066,0.192458,0.026324,0.739326,0.334667,0.049982,0.876856,0.632881
7,control_minimal__lightgbm,control_minimal,lightgbm,tree,8,-0.532941,0.773032,0.193752,0.017586,0.750622,0.301333,0.049982,0.865072,0.615702
8,always_on__logreg,always_on,logreg,linear,14,-0.532941,0.752388,0.204136,0.048336,0.729890,0.261333,0.049982,0.847568,0.595087
9,control_minimal__logreg,control_minimal,logreg,linear,8,-0.532941,0.708508,0.218617,0.043415,0.714767,0.202667,0.049982,0.811749,0.564851


## 3. Selected model metrics

In [4]:
selected_metrics = pd.read_csv(OUTPUT_DIR / 'selected_model_metrics.csv')
selected_metrics

,split,candidate_name,feature_set,model_name,feature_count,auc_ci_low,auc_ci_high,auc,brier,ece,...,precision,recall,accuracy,tp,tn,fp,fn,n_pos,n_neg,n_total
0,val,full_v2__lightgbm,full_v2,lightgbm,36,0.950817,0.959515,0.954827,0.083164,0.014494,...,0.944203,0.795333,0.870297,2386,2680,141,614,3000,2821,5821
1,id_test,full_v2__lightgbm,full_v2,lightgbm,36,0.944785,0.954154,0.949068,0.089715,0.010927,...,0.940891,0.774667,0.858787,2324,2675,146,676,3000,2821,5821
2,ood_eval,full_v2__lightgbm,full_v2,lightgbm,36,0.965858,0.969356,0.967559,0.070428,0.024139,...,0.946470,0.851701,0.899708,11917,12744,674,2075,13992,13418,27410


## 4. OOD breakdown by generator

In [5]:
ood_by_generator = pd.read_csv(OUTPUT_DIR / 'selected_model_ood_by_generator.csv')
ood_by_generator

,generator,auc,brier,ece,threshold,tpr,fpr,precision,recall,accuracy,tp,tn,fp,fn,n_pos,n_neg,n_total
0,GLIDE,0.981562,0.055757,0.046376,0.707474,0.915333,0.051394,0.949024,0.915333,0.931601,5492,5445,295,508,6000,5740,11740
1,SDv15,0.956878,0.081419,0.011266,0.707474,0.803929,0.049362,0.944297,0.803929,0.875814,6425,7299,379,1567,7992,7678,15670


## 5. Selected model importance

In [6]:
family_importance = pd.read_csv(OUTPUT_DIR / 'selected_model_family_importance.csv')
feature_importance = pd.read_csv(OUTPUT_DIR / 'selected_model_feature_importance.csv')
family_importance, feature_importance.head(15)

(                   family  importance
 0  content_adaptive_y_srm      2531.0
 1         conditional_cfa      1915.0
 2             fft_midband      1763.0
 3         control_spatial      1540.0
 4           wavelet_decay      1338.0
 5           control_color      1280.0
 6    dark_textured_hetero      1270.0
 7       control_frequency       363.0,
                            feature  importance                  family
 0               cfa_validity_score       846.0         conditional_cfa
 1                     kurt_noise_y       647.0         control_spatial
 2                spatial_snr_ratio       635.0         control_spatial
 3              energy_ratio_chroma       588.0           control_color
 4       ysrm_midtex_square5_energy       552.0  content_adaptive_y_srm
 5          ysrm_midtex_square3_mar       488.0  content_adaptive_y_srm
 6                  wav_ratio_l1_l2       482.0           wavelet_decay
 7       ysrm_midtex_square3_energy       481.0  content_adaptive_y_srm
